In [ ]:
RAG:
retrieval augumented generation
it is mainly used to retrieve the information from the external sources like pdfs,docs,txt files etc

In [ ]:
upload the file--->read the file--->convert to embeddings--->faiss--->user query-->convert query
to vector--->similarity search-->return the top results

In [ ]:
mini search engine with our own docs:
takes a .txt file(question answer file)
converts the file into embeddings(vectors)
store them in vector database--->faiss/pinecone/chroma db
user search by using any query
returns the most similar results from the context

In [ ]:
pip install sentence-transformers

In [ ]:
import streamlit as st #ui app
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

def load_model():
    return SentenceTransformer('all-MiniLM-L6-v2')

def generate_embeddings(texts,_model):
    return _model.encode(texts)

def load_documents_from_file(filepath):
    with open(filepath,'r',encoding='utf-8') as file:
        lines=[line.strip() for line in file if line.strip()]

    documents=[f"{lines[i]} {lines[i+1]}" for i in range(0,len(lines),2)]
    return documents

def create_faiss_index(_documents,_model):
    document_embeddings=generate_embeddings(_documents,_model)
    d=document_embeddings.shape[1]
    index=faiss.indexFlatL2(d)
    index.add(np.array(document_embeddings))
    return index,document_embeddings

def retrieve(query,_model,index,documents,top_k=2):
    query_embedding=generate_embeddings([query],_model)
    D,I=index.search(np.array(query_embedding),top_k)
    return [(documents[i],D[0][idx]) for idx,i in enumerate(I[0])]


def main():
    st.title("semantic search with sentence transformers + Faiss")
    st.markdown("upload a .txt file with Q&A pairs")

    uploaded_file=st.file_uploader("upload your text file", type="txt")

if uploaded_file:
    filepath="uploaded_documents.txt"
    with open(filepath,"wb") as f:
        f.write(uploaded_file.get_value())

_model=load_model()
documents=load_document_from_file(filepath)
index,_=create_faiss_index(documents,_model)


query=st.text_input("enter a query to search documents:")

if query:
    results=retrieve(query,_model,index,documents)
    st.subheader("search results")

    for doc,score in results:
        st.write(f"**score**:{score:.4f}")
        st.write(f"{doc}")

if __name__=="__main__":
    main()
